In [9]:
import torch

# Entradas bipolares en orden: 11, 10, 01 y 00
S = torch.tensor([
    [ 1,  1],
    [ 1, -1],
    [-1,  1],
    [-1, -1]
], dtype=torch.float32)

# NAND: 0, 1, 1, 1
T_NAND = torch.tensor(
    [-1, 1, 1, 1],
    dtype=torch.float32
)

# NOR: 0, 0, 0, 1
T_NOR = torch.tensor(
    [-1, -1, -1, 1],
    dtype=torch.float32
)

In [10]:
def activar(x, w, b, theta=0):
    """Calcula la salida bipolar de una neurona."""
    
    y_in = torch.dot(x.float(), w) + b

    # Activación usada por Hebb
    if theta == 0:
        return 1 if y_in >= 0 else -1

    # Activación simétrica usada por el perceptrón
    if y_in > theta:
        return 1
    if y_in < -theta:
        return -1

    return 0


def entrenar_hebb(X, T):
    """Obtiene pesos y bias aplicando la regla de Hebb."""
    
    w = (X * T[:, None]).sum(dim=0)
    b = T.sum()

    return w, b


def entrenar_perceptron(
    X,
    T,
    alpha=1,
    theta=1,
    max_epochs=100
):
    """Entrena un perceptrón hasta completar una época sin errores."""

    w = torch.zeros(X.shape[1])
    b = torch.tensor(0.0)

    for epoch in range(1, max_epochs + 1):

        errores = 0

        for x, t in zip(X, T):

            y = activar(x, w, b, theta)

            if y != int(t.item()):
                w += alpha * x * t
                b += alpha * t
                errores += 1

        if errores == 0:
            return w, b, epoch

    return w, b, max_epochs

In [11]:
# NAND mediante Hebb
w_hebb, b_hebb = entrenar_hebb(
    S,
    T_NAND
)

# NAND mediante perceptrón
w_per, b_per, epocas = entrenar_perceptron(
    S,
    T_NAND,
    alpha=1,
    theta=1
)

print("NAND con Hebb")
print("Pesos:", w_hebb.tolist())
print("Bias:", b_hebb.item())

print("\nNAND con perceptrón")
print("Pesos:", w_per.tolist())
print("Bias:", b_per.item())
print("Épocas:", epocas)

NAND con Hebb
Pesos: [-2.0, -2.0]
Bias: 2.0

NAND con perceptrón
Pesos: [-2.0, -2.0]
Bias: 2.0
Épocas: 3


In [12]:
def a_binario(valor):
    return 1 if int(valor) == 1 else 0


def probar_modelo(nombre, funcion):
    print(f"\n{name}")
    print("A  B  Esperado  Obtenido")

    aciertos = 0

    for x, t in zip(S, T_NAND):

        y = funcion(x)

        a, b = [a_binario(v) for v in x]
        esperado = a_binario(t)
        obtenido = a_binario(y)

        aciertos += esperado == obtenido

        print(
            f"{a}  {b}      "
            f"{esperado}         {obtenido}"
        )

    print(f"Exactitud: {aciertos / len(S) * 100:.0f}%")

In [13]:
w_nor, b_nor = entrenar_hebb(
    S,
    T_NOR
)

print("Pesos NOR:", w_nor.tolist())
print("Bias NOR:", b_nor.item())

Pesos NOR: [-2.0, -2.0]
Bias NOR: -2.0


In [14]:
def nor(a, b):
    entrada = torch.tensor(
        [a, b],
        dtype=torch.float32
    )

    return activar(
        entrada,
        w_nor,
        b_nor
    )


def nand_desde_nor(x):
    a, b = x.tolist()

    n1 = nor(a, a)        # NOT A
    n2 = nor(b, b)        # NOT B
    n3 = nor(n1, n2)      # A AND B
    q = nor(n3, n3)       # NAND

    return q

In [15]:
print("A  B  NOT A  NOT B  AND  NAND")

for x in S:

    a, b = x.tolist()

    n1 = nor(a, a)
    n2 = nor(b, b)
    n3 = nor(n1, n2)
    q = nor(n3, n3)

    valores = [a, b, n1, n2, n3, q]
    valores = [a_binario(v) for v in valores]

    print(*valores, sep="   ")

A  B  NOT A  NOT B  AND  NAND
1   1   0   0   1   0
1   0   0   1   0   1
0   1   1   0   0   1
0   0   1   1   0   1
